In [3]:
from src.server.models.session_model import USER_PERMISSION__PORTAL
from src.utils.hash_utils import sha256
from src.server.models.user.UserModel import UserModel
from src.db.mysql import DbConnection
from src.server.models.billing.billing_data import BillingUtils
from src.utils.gembase_utils import GembaseUtils
from src import external_api

sheet_id = "1VpZ24bAZ84IJI-0nsX0VxcZOk0KB4qD_s8u95f8t3rw"
range = "Clients"
data = external_api.read_sheet(sheet_id, range, True)

users_data = []
org_data = {}
domain_per_dev = {}

for it in data:
    if it["Email"] != "jukka.hilvonen@nitrogames.fi":
        continue
    if it["Email"] == "":
        continue
    if it["Company ID"] == "":
        continue

    u = {}
    users_data.append(u)

    u["email"] = it["Email"]
    u["name"] = it["Name"]
    u["dev_id_int"] = int(it["Company ID"])
    dev_id_int = u["dev_id_int"]
    if dev_id_int not in org_data:
        org_data[dev_id_int] = {
            "modules": {},
            "users": []
        }
    u["admin"] = int(it["Admin"])
    org_data[dev_id_int]["users"].append({
        "email": it["Email"],
        "role": "admin" if u["admin"] == 1 else "member"
    })

    if u["admin"] == 1:
        email_domain = GembaseUtils.get_email_domain(it["Email"])
        u["email_domain"] = email_domain
        domain_per_dev[dev_id_int] = u["email_domain"]
        org_data[dev_id_int]["email_domain"] = email_domain
    m = []
    if it["Auditor"] == "1":
        m.append(BillingUtils.MODULE_AUDITOR)
    if it["Tuner"] == "1":
        m.append(BillingUtils.MODULE_TUNER)
    if it["Gaps"] == "1":
        m.append(BillingUtils.MODULE_GAPS)
    if it["Studies"] == "1":
        m.append(BillingUtils.MODULE_STUDIES)
    if it["Players"] == "1":
        m.append(BillingUtils.MODULE_PLAYERS_EXPLORER)
    if it["Games"] == "1":
        m.append(BillingUtils.MODULE_GAMES_EXPLORER)
    if it["Lenses"] == "1":
        m.append(BillingUtils.MODULE_LENSES)
    if it["Creator"] == "1":
        m.append(BillingUtils.MODULE_CREATOR)

    u["modules"] = m
    for module_id in m:
        if module_id not in org_data[dev_id_int]["modules"]:
            org_data[dev_id_int]["modules"][module_id] = 0
        org_data[dev_id_int]["modules"][module_id] += 1

print(org_data)
print(users_data)

{35115: {'modules': {}, 'users': [{'email': 'jukka.hilvonen@nitrogames.fi', 'role': 'admin'}], 'email_domain': 'nitrogames.fi'}}
[{'email': 'jukka.hilvonen@nitrogames.fi', 'name': 'Jukka Hilvonen', 'dev_id_int': 35115, 'admin': 1, 'email_domain': 'nitrogames.fi', 'modules': []}]


In [8]:
from src.runtime_constants import rr
from src.server.models.billing.billing_model import BillingModel
from src.server.models.organization.organization_model import OrganizationModel

#conn = DbConnection(rr.ENV_PROD, is_remote=True)
conn = DbConnection()


for dev_id_int in org_data:

    if "email_domain" not in org_data[dev_id_int]:
        continue

    users = org_data[dev_id_int]["users"]

    row_org_id = conn.select_one_or_none("""
    SELECT od.organization_id
      FROM app.organization_domains od
     WHERE od.domain = %s
    """, [org_data[dev_id_int]["email_domain"]])

    if row_org_id is not None:
        org_id = row_org_id["organization_id"]

        OrganizationModel.s__delete_organization(
            conn=conn,
            organization_id=org_id
        )

        row_billing_id = conn.select_one_or_none("""
        SELECT b.id
          FROM billing.billings b
         WHERE b.organization_id = %s
        """, [org_id])

        if row_billing_id is not None:

            BillingModel.s__delete(
                conn=conn,
                billing_id=row_billing_id["id"]
            )

    modules = []
    for module_id in org_data[dev_id_int]["modules"]:
        modules.append({
            "id": module_id,
            "seats": org_data[dev_id_int]["modules"][module_id]
        })

    if len(modules) == 0:
        continue

    prime_number = conn.select_one("""
    SELECT p.id
      FROM app.def_prime_numbers p
     ORDER BY RAND()
     LIMIT 1
    """)["id"]

    org_id = OrganizationModel.s__create_organization(
        conn=conn,
        dev_id_int=dev_id_int,
        prime_number=prime_number,
        users=[],
        email_domain=org_data[dev_id_int]["email_domain"],
        credits_value=25
    )

    billing_id = BillingModel.s__create(
        conn=conn,
        organization_id=org_id,
        user_id=0,
        payment_id=0,
        expire_months=12,
        modules=modules
    )

conn.commit()
conn.close()

In [5]:
from src.server.models.user.UserHelper import UserHelper
from src.server.models.user_constants import UserConstantsClass
from src.runtime_constants import rr

conn = DbConnection(rr.ENV_PROD, is_remote=True)
#conn = DbConnection()

temp_user = 1

admin_id = conn.select_one("""
SELECT u.id FROM app.users u WHERE u.guid = %s
""", [UserConstantsClass.ADMIN_USER_GUID])["id"]

for user in users_data:

    conn.query("""
        DELETE FROM app.users_registration_requests r
         WHERE r.temp_user = 1
           AND r.email = %s
        """, [user["email"]])

    conn.query("""
        DELETE FROM app.organization_requests_modules rm
        WHERE rm.request_id IN (
        SELECT r.request_guid
          FROM app.organization_requests r
         WHERE r.temp_user = 1
           AND r.email = %s
           )
        """, [user["email"]])
    conn.query("""
        DELETE FROM app.organization_requests r
         WHERE r.temp_user = 1
           AND r.email = %s
        """, [user["email"]])

    if len(user["modules"]) == 0:
        conn.query("""
        INSERT INTO app.users_registration_requests (guid, email, name, dev_id_int, temp_user)
        VALUES (uuid(), %s, %s, %s, %s)
        """, [user["email"], user["name"], user["dev_id_int"], temp_user])
    else:
        org_id = conn.select_one("""
        SELECT o.id
          FROM app.organization o,
               app.organization_domains od
         WHERE o.dev_id_int = %s
           AND o.id = od.organization_id
           AND od.domain = %s
        """, [user["dev_id_int"], domain_per_dev[user["dev_id_int"]]])["id"]
        billing_id = conn.select_one("""
        SELECT b.id
          FROM billing.billings b
         WHERE b.organization_id = %s
        """, [org_id])["id"]
        role = "admin" if user["admin"] == 1 else "member"
        email_id = UserHelper.get_or_create_email_id(conn=conn, email=user["email"])
        req_id = conn.insert("""
        INSERT INTO app.organization_requests (organization_id, email, request_guid, from_user_id, role, name, temp_user)
        VALUES (%s, %s, uuid(), %s, %s, %s, %s)
        """, [org_id, user["email"], admin_id, role, user["name"], temp_user])
        for module_id in user["modules"]:
            conn.insert("""
            INSERT INTO app.organization_requests_modules (request_id, billing_id, module_id)
            VALUES (%s, %s, %s)
            """, [req_id, billing_id, module_id])

conn.commit()
conn.close()

In [10]:
conn.close()